# Initial Datahandeling

### This notebook shows the initial handling of utterances in the Danish Parliament.
### It includes the following:

- Loading the two overlapping datasets of utterences in the Danish Parliament
    - Due to the ParlLawSpeech data being in .rds format, it was read into r, converted to .csv format and then saved in this repo
- Merging them by end data of corp_v2
- Sentence segmentation
- Reformatting to json object for easier handling during labeling for validation, training and inference
- Datacleaning
- Splitting into training data (including validationset to be extracted), and inference data

In [3]:
# Load packages

import pandas as pd
import os
from danish_minister_party_pipeline import assign_party

# Corp_Folketing_V2

In [8]:
#read the corp folketing 

corp_v2_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "csv_meetings",
                            "Corp_Folketing_V2.csv")
'''
corp_v2_path = os.path.join("..",
                            "..",
                            "raw_data",
                            "Corp_Folketing_V2.csv")
'''
# Read the two datasets
df_corp = pd.read_csv(corp_v2_path)#, index_col=0)
#x = df_corp.pop("Unnamed: 0") #dummy variable

#make date a datetime object
df_corp["date"] = pd.to_datetime(df_corp["date"])

In [9]:
df_corp.head()

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country
0,1997-10-07,Dagsorden,1,Gert Petersen,NaN,NaN,True,191,Mødet er åbnet. I henhold til grundloven er Fo...,DK-Folketing,DNK
1,1997-10-07,Dagsorden,2,Formanden,NaN,NaN,True,182,"Jeg vil gerne takke Tinget for den tillid, man...",DK-Folketing,DNK
2,1997-10-07,Statsministerens redegørelse i henhold til gru...,3,Poul Nyrup Rasmussen,S,379.0,False,18662,For 25 år siden sagde et flertal i befolkninge...,DK-Folketing,DNK
3,1997-10-09,1) Indstilling fra Udvalget til Valgs Prøvelse.,2,Formanden,NaN,NaN,True,47,Fra Udvalget til Valgs Prøvelse har jeg modtag...,DK-Folketing,DNK
4,1997-10-09,2) Forhandling om redegørelse nr. R 1.,3,Torben Lund,S,379.0,False,2865,Vi står over for en meget afgørende folketings...,DK-Folketing,DNK


In [10]:
#This dataset spans the following time period

min_time_corp = df_corp["date"].min()
max_time_corp = df_corp["date"].max()

print(f"The Corp dataset has coverage from {min_time_corp.date()} to {max_time_corp.date()}")

The Corp dataset has coverage from 1997-10-07 to 2018-12-20


# XML speeches

In [12]:
#read the PLS data. 
# OBS has been made to .csv in file ./PLS_rds_to_csv.RMD

'''xml_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "Data_outside_git",
                            "meetings.csv")'''

xml_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "csv_meetings",
                            "meetings.csv")

# Read the two datasets
df_xml = pd.read_csv(xml_path)

#make date a datetime object
df_xml["date"] = pd.to_datetime(df_xml["date"])

In [13]:
df_xml.head()

,paragraph_nr,date,speaker,party,role,text,source_file,chair
0,0,2010-05-31,Holger K. Nielsen,SF,formand,Mødet er åbnet. Finansudvalget har afgivet: Be...,20091_M100_helemoedet.xml,True
1,1,2010-05-31,Holger K. Nielsen,SF,formand,Forhandlingen er åbnet. Fru Ulla Tørnæs som or...,20091_M100_helemoedet.xml,True
2,2,2010-05-31,Ulla Tørnæs,V,medlem,Regeringen har sammen med Dansk Folkeparti ind...,20091_M100_helemoedet.xml,False
3,3,2010-05-31,Holger K. Nielsen,SF,formand,Det tror jeg bestemt der gør. Der er en række ...,20091_M100_helemoedet.xml,True
4,4,2010-05-31,Torben Hansen,S,medlem,"Tak, hr. formand. Det skal ikke være nogen hem...",20091_M100_helemoedet.xml,False


In [14]:
#This dataset spans the following time period

min_time_xml = df_xml["date"].min()
max_time_xml = df_xml["date"].max()

print(f"The xml dataset has coverage from {min_time_xml.date()} to {max_time_xml.date()}")

The xml dataset has coverage from 2009-10-06 to 2026-02-26


# Make the XML speeches match the format of ParlSpeech

To do so, the following must be done:

- Chair column as TRUE/FALSE - DONE
- Missing values for all non-chairmen to be inferred
- Make sure the two datasets has matching column structure



In [15]:
# Keep only rows in df2 strictly after max date of df1
df_xml_filtered = df_xml[df_xml['date'] > max_time_corp]

no_party = df_xml_filtered.loc[df_xml_filtered["party"].isna(), "speaker"]
no_party_unique = no_party.unique()

print(no_party_unique) # Print the names of the speakers that have no party affiliation
print(len(no_party))    # print the expected number of rows to be modified

<StringArray>
[    'Claus Hjort Frederiksen',               'Rasmus Jarlov',
            'Anders Samuelsen',          'Søren Pape Poulsen',
                 'Mai Mercado',    'Lars Christian Lilleholt',
           'Ellen Trane Nørby',             'Merete Riisager',
       'Jakob Ellemann-Jensen',                'Tommy Ahlers',
        'Lars Løkke Rasmussen',           'Karsten Lauritzen',
         'Troels Lund Poulsen',                           ' ',
              'Inger Støjberg',             'Kristian Jensen',
             'Ole Birk Olesen', 'Simon Emil Ammitzbøll-Bille',
                 'Ulla Tørnæs',             'Eva Kjer Hansen',
                 'Thyra Frank',                  'Mette Bock',
                 'Astrid Krag',               'Trine Bramsen',
               'Mogens Jensen',            'Kaare Dybvad Bek',
           'Mette Frederiksen',              'Nicolai Wammen',
             'Magnus Heunicke',               'Nick Hækkerup',
               'Dan Jørgensen',  'Pernill

In [16]:
df_xml_w_parties = assign_party(df_xml_filtered)

[assign_party]  Total rows          : 374224
[assign_party]  Pre-filled (skipped): 345427
[assign_party]  Newly matched       : 28350
[assign_party]  Unmatched / empty   : 0


In [17]:
#check if all are filled
no_party = df_xml_w_parties.loc[df_xml_w_parties["party"].isna(), "speaker"]
no_party_unique = no_party.unique()

print(no_party_unique) # Print the names of the speakers that have no party affiliation
print(len(no_party)) 

<StringArray>
[' ']
Length: 1, dtype: str
447


# OBS there are 447 paragraphs witout speaker or party. Why?

See header here for the fist 5

In [18]:
df_xml_w_parties.loc[df_xml_w_parties["party"].isna()].head()

,paragraph_nr,date,speaker,party,role,text,source_file,chair
440320,138,2019-01-16,,NaN,NaN,(Spørgsmålet er efter ønske fra ministeren udg...,20181_M46_helemoedet.xml,False
440321,139,2019-01-16,,NaN,NaN,(Spørgsmålet er efter ønske fra ministeren udg...,20181_M46_helemoedet.xml,False
447396,122,2019-02-20,,NaN,NaN,"(Spørgsmålet er udgået, da det er taget tilbag...",20181_M62_helemoedet.xml,False
447431,157,2019-02-20,,NaN,NaN,(Spørgsmålet er udsat til næste spørgetid).,20181_M62_helemoedet.xml,False
447449,175,2019-02-20,,NaN,NaN,(Spørgsmålet er overgået til skriftlig besvare...,20181_M62_helemoedet.xml,False


In [19]:
headers_maybe = df_xml_w_parties.loc[df_xml_w_parties["party"].isna(), "text"]

for text in headers_maybe[:10]:
    print(text)

(Spørgsmålet er efter ønske fra ministeren udgået af dagsordenen og udsat til spørgetiden onsdag den 30. januar 2019).
(Spørgsmålet er efter ønske fra ministeren udgået af dagsordenen og udsat til spørgetiden onsdag den 30. januar 2019).
(Spørgsmålet er udgået, da det er taget tilbage af spørgeren).
(Spørgsmålet er udsat til næste spørgetid).
(Spørgsmålet er overgået til skriftlig besvarelse).
(Spørgsmålet er overgået til skriftlig besvarelse).
(Spørgsmålet er overgået til skriftlig besvarelse).
(Spørgsmålet er udgået under henvisning til Folketingets forretningsordens § 20, stk. 5).
(Spørgsmålet er udgået under henvisning til Folketingets forretningsordens § 20, stk. 5).
(Spørgsmålet er udgået under henvisning til Folketingets forretningsordens § 20, stk. 5).


In [20]:
# Deleting entries without any party affiliation
df_xml_w_parties = df_xml_w_parties.dropna(subset=["party"])

#dropping irrelevant columns before merge
df_xml_w_parties = df_xml_w_parties.drop(columns=["paragraph_nr", "role", "source_file"])

# Merging datasets

In [21]:
#rename column(s) in each to match the counterpart

print(df_xml_w_parties.columns)
print(df_corp.columns) # party.facts.id to partyfacts_ID

Index(['date', 'speaker', 'party', 'text', 'chair'], dtype='str')
Index(['date', 'agenda', 'speechnumber', 'speaker', 'party', 'party.facts.id',
       'chair', 'terms', 'text', 'parliament', 'iso3country'],
      dtype='str')


In [22]:
# Dropping irrelevant columnds before merge

df_corp = df_corp.drop(columns=["party.facts.id", "parliament", "iso3country", "terms", "agenda", "speechnumber"])
print(df_corp.columns) #double check

Index(['date', 'speaker', 'party', 'chair', 'text'], dtype='str')


In [23]:

# Concatenate
df_merged = pd.concat([df_corp, df_xml_w_parties], ignore_index=True)

# sort by date
df_merged = df_merged.sort_values('date').reset_index(drop=True)

In [24]:
#check that the span now is the entire period
#This dataset spans the following time period

min_time_merged = df_merged["date"].min()
max_time_merged = df_merged["date"].max()

print(f"The merged datasets has coverage from {min_time_merged.date()} to {max_time_merged.date()}")

The merged datasets has coverage from 1997-10-07 to 2026-02-26


In [26]:
len(df_xml_w_parties)
len(df_merged)

1145957

### As the future analysis will discard any utterences from the chair of parliament, as these serve to aid order and time of speaking, chair == True will be discarded now for efficiency reasons. Additionally, they typically speak for a very long time at the beginning fo each recorded session

In [27]:

df_merged = df_merged[df_merged['chair'] == False] # Only include chair == False

'''

df_merged = df_merged.loc[
    (df_merged["chair"].isna()) | ((df_merged["chair"] == "False"))]

df_merged = df_merged[df_merged['role'] != "formand"] # Same idea, formatted differently in the xml dataset
'''


'\n\ndf_merged = df_merged.loc[\n    (df_merged["chair"].isna()) | ((df_merged["chair"] == "False"))]\n\ndf_merged = df_merged[df_merged[\'role\'] != "formand"] # Same idea, formatted differently in the xml dataset\n'

In [28]:
len(df_merged)

638953

In [29]:
df_merged

,date,speaker,party,chair,text
2,1997-10-07,Poul Nyrup Rasmussen,S,False,For 25 år siden sagde et flertal i befolkninge...
3,1997-10-09,Frank Dahlgaard,KF,False,»De Konservative kræver den sorte skole tilbag...
4,1997-10-09,Kristian Thulesen Dahl,DF,False,Jeg må give hr. Holger K. Nielsen fuldstændig ...
5,1997-10-09,Holger K. Nielsen,SF,False,"Til hr. Frank Dahlgaard kan jeg sige, at når m..."
6,1997-10-09,Frank Dahlgaard,KF,False,"Det er jo forkert, hvad hr. Holger K. Nielsen ..."
...,...,...,...,...,...
1145948,2026-02-26,Benny Engelbrecht,S,False,"Igen er vi tilbage til det ganske enkle, at fr..."
1145949,2026-02-26,Lotte Rod,RV,False,Jeg vil gerne spørge igen: Hvorfor har Sociald...
1145951,2026-02-26,Benny Engelbrecht,S,False,"Jeg tror ikke, at de mange førtidspensionister..."
1145953,2026-02-26,Lotte Rod,RV,False,Men hvorfor vælger Socialdemokraterne ikke at ...


In [30]:
#Save the data

merged_data_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "csv_meetings"
                            "merged_speech_data.csv")
                                
df_merged.to_csv(merged_data_path, index=False)